# Stage D: targeted scheduler optimization

Fixed DeepSeek-V4-Flash-0731 / vLLM 0.33.0 / GPUs 0–3. The historical A/B/C baselines remain immutable. This notebook separates profiled mechanism evidence, unprofiled matched measurements, stability, resources, and narrow natural-EOS quality probes. Two independent restart blocks are shown individually; this is not a maximum-capacity certificate.


In [ ]:
from pathlib import Path
import json, hashlib, sys
D = next(p/'research/dsv4-flash0731/scheduler' for p in [Path.cwd(),*Path.cwd().parents] if (p/'research/dsv4-flash0731/scheduler').is_dir())
summary_path=D/'summary/performance-final-r1.json'
profile_path=D/'summary/profile-matched-r1.json'
summary=json.loads(summary_path.read_text()); profile=json.loads(profile_path.read_text())
assert summary['status']==profile['status']=='PASS'
assert not summary['errors']
print('Performance SHA256:',hashlib.sha256(summary_path.read_bytes()).hexdigest())
print('Profile SHA256:',hashlib.sha256(profile_path.read_bytes()).hexdigest())
print('Per-window rule:',summary['rules']['version'])


## Profile mechanism and failed experiment

Profile latencies are excluded from normal rankings. d02 failed at the original 120 s export deadline despite completing its requests; d02r2 changes only the export wait to 600 s. Raw failed evidence is retained. Scheduler call indices and optimistic token counters are not GPU completion timestamps.


In [ ]:
for e in profile['experiments']:
    print(e['experiment'],e['status'],'requests',e['profile_request_count'])
failed=json.loads((D/'results/scheduler-tp4-fairshare-dense-profile/status.json').read_text())
assert failed['supervisor_exit_code'] != 0 and failed['source_identity_unchanged']
print('Retained d02 failure:',failed['supervisor_exit_code'])
print('Mechanism figure: figures/profile-matched-r1/request-allocations.png')

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
figure_path=D/'figures/profile-matched-r1/request-allocations.png'
plt.figure(figsize=(12,8));plt.imshow(mpimg.imread(figure_path));plt.axis('off');plt.show()


## Unprofiled measurements

Observe is the original scheduling policy through the same research wrapper, not an unwrapped native baseline. FairShare adds ordering and a dynamic per-request cap; tracing is disabled in both. All shape percentiles retain their sample counts. TTFT and E2E start at the planned arrival; TPOT is a mean output interval, not ITL.


In [ ]:
sys.path.insert(0,str(D/'scripts'))
import plot_performance_d as plotting
rows=plotting.collect(summary)
assert len(rows)==12
for row in rows:
    print(row['experiment'],row['phase'],'short n=',row['short_requests'],
          'TTFT P95=',round(row['short_ttft_p95_s'],3),
          'TPOT P95 ms=',round(row['short_tpot_p95_ms'],2),
          'short Goodput=',round(row['short_goodput'],4),
          '64K TTFT P95=',round(row['long_ttft_p95_s'],3))
print('Request inventory:',sum(s['offered_requests'] for s in summary['stages']))
print('Formal requests:',sum(s['offered_requests'] for s in summary['stages'] if s['included_as_measurement']))
assert len(summary['paired_comparisons'])==6
assert all(p['status']=='PASS' for p in summary['paired_comparisons'])


In [ ]:
import matplotlib.pyplot as plt
figs=plotting.figures(rows)
plt.show()
print('Rendered two figures from the same 12 source rows; no profiled latency included.')


## Stability and resource boundaries

Report complete cohort horizon plus drain. A fixed burst is not sustained offered-rate evidence. 32K/64K joint TTFT/E2E SLO remains undefined. GPU energy refers to sampled boards, not wall-plug power; device busy is not SM occupancy. No new production SLO or global maximum capacity is asserted.


In [ ]:
for row in rows:
    print(row['experiment'],row['phase'],'stability=',row['stability'],
          'energy kJ=',row['energy_kj'],'KV peak %=',row['kv_peak_pct'],
          'preemptions=',row['preemptions'],'drain s=',row['drain_s'])
for record in summary['repeated_window_observations']:
    assert record['capacity_certified'] is False
    print(record['policy'],record['classifications'],
          'all supplied stable=',record['all_supplied_long_windows_stable'],
          'short P95 all supplied=',record['short_p95_slo_all_supplied'])


## Independent quality probes

Eight synthetic arithmetic, JSON, and history questions, repeated twice per server with natural EOS. Incorrect answers do not retract separately measured throughput. This small sequential probe does not establish broad model quality, concurrent long-context answer quality, or Agent/SWE-bench capability.


In [ ]:
for run in summary['runs']:
    q=run['quality']
    print(run['experiment'],'quality audit=',q['status'],
          'protocol successes=',q.get('protocol_successes'),
          'requests=',q.get('requests'),'strict answers passed=',q.get('quality_passed'))
    for case,counts in sorted(q.get('by_case',{}).items()): print(' ',case,counts)
assert all(run['quality']['excluded_from_performance'] for run in summary['runs'])
